# 03 · Feature engineering

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.

Shows the feature registry (`configs/features.yaml`), the fitted feature pipelines and their fit-scope records (training rows only), and the resulting matrices per set. Leakage guards are enforced by `tests/test_leakage.py`.

In [ ]:
from pathlib import Path

import json
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

from aml_triage.config import load

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load(ROOT / "configs" / "base.yaml")
REPORTS = ROOT / cfg.paths.reports_dir
PROCESSED = ROOT / cfg.paths.processed_dir
print("config hash:", cfg.config_hash())

## Feature registry

In [ ]:
from aml_triage.features.base import load_registry

reg = load_registry(ROOT / cfg.features.registry)
display(pd.DataFrame([{"name": d.name, "kind": d.kind, "availability": d.available_at_prediction_time, "sets": ", ".join(d.sets), "rationale": d.rationale} for d in reg]))

## Fit scope and feature lists per set

Every fitted transform (one-hot categories, amount bucket edges) must have been fitted on `train` only.

In [ ]:
rows = []
for s in ["primary", "strict_pretx", "posttx_ablation"]:
    fs = PROCESSED / f"feature_pipeline_{s}.fitscope.json"
    fl = PROCESSED / f"features_{s}.json"
    if fs.exists() and fl.exists():
        rec = json.loads(fs.read_text()); lst = json.loads(fl.read_text())
        rows.append({"set": s, "fitted_on": rec["fitted_on"], "transformed_on": rec["transformed_on"], "n_features": len(lst["features"]), "batch_only": ", ".join(lst["batch_only"]) or "-", "aggregates": lst["aggregates_included"]})
display(pd.DataFrame(rows) if rows else Markdown("_Run `python -m aml_triage build-features --feature-set primary` first._"))

## Sample of the primary training matrix (feature columns only; identifiers are never present)

In [ ]:
from aml_triage.features.pipeline import load_feature_matrix

p = PROCESSED / "features_primary_train.parquet"
if p.exists():
    X, meta = load_feature_matrix(PROCESSED, "primary", "train")
    display(X.describe().T)
    print("meta columns:", list(meta.columns))
else:
    display(Markdown("_Feature matrices not built yet._"))

## Data dictionary (raw and engineered)

In [ ]:
display(Markdown((REPORTS / "data_dictionary.md").read_text()))